In [16]:
import pyspark
from pyspark.sql import SparkSession
from neo4j import GraphDatabase
from utils import load_config

class Neo4jLoader:
    def __init__(self, config_file='config.json'):
        self.config = load_config(config_file)
        self.uri = self.config.get("neo4j_uri", "neo4j+s://7bb39fe0.databases.neo4j.io")
        self.user = self.config.get("neo4j_user", "7bb39fe0")
        self.password = self.config.get("neo4j_pass", "5wCN9zh1-kH5f4ZODGGvU4V0i-DyreUtxfBHkjjIjis")
        
        self.driver = GraphDatabase.driver(self.uri, auth=(self.user, self.password))
        self.spark = self.setup_spark_session()

    def setup_spark_session(self):
        session = (SparkSession.builder
            .appName("Neo4jLoader")
            .getOrCreate())
        session.sparkContext.setLogLevel("WARN")
        return session

    def transfer_kitchen_data(self):
        print("Reading kitchen data with Spark...")
        kitchen_path = self.config.get("kitchen_data_path", "./output/kitchen_data")
        
        try:
            kitchen_df = self.spark.read.parquet(kitchen_path)
            records = [row.asDict(recursive=True) for row in kitchen_df.collect()]
            
            if records:
                print("Sending kitchen events to Neo4j Aura...")
                query = """
                UNWIND $events AS event
                MERGE (b:Batch {id: toString(event.batch_id)})
                MERGE (s:Station {id: toString(event.station_id)})
                MERGE (r:Recipe {id: toString(event.recipe_id)})
                MERGE (s)-[rel:PROCESSED]->(b)
                SET rel.action = event.action,
                    rel.time = event.event_timestamp,
                    rel.temp_celsius = event.temperature_celsius,
                    rel.weight = event.weight_kg
                MERGE (b)-[:USES_RECIPE]->(r)
                
                WITH event, r
                WHERE event.ingredients IS NOT NULL
                UNWIND event.ingredients AS ing
                MERGE (i:Ingredient {name: toString(ing.item)})
                MERGE (r)-[req:REQUIRES]->(i)
                """
                with self.driver.session() as session:
                    session.run(query, events=records)
                print(f"Success. Inserted {len(records)} kitchen events.")
            else:
                print("No kitchen data found.")
                
        except Exception as error:
            print(f"Failed to move kitchen data. Error: {error}")

    def transfer_dispatch_data(self):
        print("\nReading dispatch data with Spark...")
        dispatch_path = self.config.get("dispatch_data_path", "./output/dispatch_data")
        
        try:
            dispatch_df = self.spark.read.parquet(dispatch_path)
            records = [row.asDict(recursive=True) for row in dispatch_df.collect()]
            
            if records:
                print("Sending dispatch events to Neo4j Aura...")
                query = """
                UNWIND $events AS event
                MERGE (b:Batch {id: toString(event.batch_id)})
                MERGE (d:Driver {id: toString(event.driver_id)})
                MERGE (c:Canteen {id: toString(event.canteen_id)})
                MERGE (d)-[rel:DELIVERS]->(b)
                SET rel.action = event.action,
                    rel.time = event.event_timestamp,
                    rel.truck_temp_celsius = event.truck_temp_celsius
                MERGE (b)-[:DESTINED_FOR]->(c)
                """
                with self.driver.session() as session:
                    session.run(query, events=records)
                print(f"Success. Inserted {len(records)} dispatch events.")
            else:
                print("No dispatch data found.")
                
        except Exception as error:
            print(f"Failed to move dispatch data. Error: {error}")

    def run_transfer(self):
        self.transfer_kitchen_data()
        self.transfer_dispatch_data()
        print("\nClosing connections...")
        self.spark.stop()
        self.driver.close()

if __name__ == "__main__":
    app = Neo4jLoader()
    app.run_transfer()

26/03/29 00:19:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Reading kitchen data with Spark...
Sending kitchen events to Neo4j Aura...
Success. Inserted 159 kitchen events.

Reading dispatch data with Spark...
Sending dispatch events to Neo4j Aura...
Success. Inserted 160 dispatch events.

Closing connections...


In [19]:
from neo4j import GraphDatabase
from utils import load_config
from datetime import datetime

class Neo4jQueries:
    def __init__(self, config_file='config.json'):
        self.config = load_config(config_file)
        self.uri = self.config.get("neo4j_uri", "neo4j+s://7bb39fe0.databases.neo4j.io")
        self.user = self.config.get("neo4j_user", "7bb39fe0")
        self.password = self.config.get("neo4j_pass", "5wCN9zh1-kH5f4ZODGGvU4V0i-DyreUtxfBHkjjIjis")
        
        self.driver = GraphDatabase.driver(self.uri, auth=(self.user, self.password))

    def track_ingredient_recall(self, target_ingredient, target_date):
        print(f"\n--- TRACKING RECALL FOR {target_ingredient} ON {target_date} ---")
        query = """
        MATCH (i:Ingredient {name: $ingredient})<-[:REQUIRES]-(r:Recipe)<-[:USES_RECIPE]-(b:Batch)-[:DESTINED_FOR]->(c:Canteen)
        MATCH (s:Station)-[p:PROCESSED]->(b)
        WHERE p.time STARTS WITH $date
        RETURN DISTINCT i.name AS Target_Ingredient, r.id AS Recipe_Used, b.id AS Batch_ID, c.id AS Affected_Canteen
        ORDER BY c.id
        """
        
        with self.driver.session() as session:
            results = session.run(query, ingredient=target_ingredient, date=target_date).data()
            
        if not results:
            print(f"No records found for ingredient '{target_ingredient}' on {target_date}.")
            return
            
        print(f"Found {len(results)} affected batches.")
        for record in results:
            recipe = record.get('Recipe_Used')
            batch = record.get('Batch_ID')
            canteen = record.get('Affected_Canteen')
            print(f"- {batch} (Recipe: {recipe}) delivered to Canteen {canteen}")

    def map_contamination_risk(self, target_station, target_date):
        print(f"\n--- MAPPING RISK FOR STATION {target_station} ON {target_date} ---")
        query = """
        MATCH (s:Station {id: $station})-[p1:PROCESSED]->(b:Batch)
        WHERE p1.time STARTS WITH $date
        OPTIONAL MATCH (later_s:Station)-[p2:PROCESSED]->(b)
        WHERE p2.time > p1.time
        WITH s, b, collect(DISTINCT later_s.id) AS Later_Stations
        MATCH (drv:Driver)-[:DELIVERS]->(b)-[:DESTINED_FOR]->(c:Canteen)
        RETURN s.id AS Risk_Station, b.id AS Shared_Batch, Later_Stations, drv.id AS Exposed_Driver, c.id AS Affected_Canteen
        ORDER BY b.id
        """
        
        with self.driver.session() as session:
            results = session.run(query, station=target_station, date=target_date).data()
            
        if not results:
            print(f"No connections found for station '{target_station}' on {target_date}.")
            return
            
        print(f"\nFound {len(results)} exposed batches and their respective paths.")
        for record in results:
            batch = record.get('Shared_Batch')
            later_stations = record.get('Later_Stations')
            driver = record.get('Exposed_Driver')
            canteen = record.get('Affected_Canteen')
            
            if later_stations:
                stations_text = ", ".join(later_stations)
            else:
                stations_text = "None"
                
            print(f"- {batch} -> Later Stations: [{stations_text}] delivered by Driver {driver} to Canteen {canteen}")

    def run_menu(self):        
        while True:
            print("\n" + "="*30)
            print("       NEO4J QUERY MENU")
            print("="*30)
            print("1. Track Ingredient Recall")
            print("2. Map Station Contamination Risk")
            print("3. Exit")
            choice = input("\nEnter your choice (1/2/3): ").strip()
            
            if choice == '3':
                print("Exiting the query tool.")
                break
                
            if choice in ['1', '2']:
                while True:
                    user_date = input("Enter the date to search (Format: YYYY-MM-DD): ").strip()
                    try:
                        datetime.strptime(user_date, "%Y-%m-%d")
                        break
                    except ValueError:
                        print("Error: The date must be in YYYY-MM-DD format. Please try again.")

                if choice == '1':
                    ingredient_input = None
                    while not ingredient_input:
                        ingredient_input = input("\nEnter the ingredient name to track (e.g., Chicken): ").strip()
                        if not ingredient_input:
                            print("Error: The input cannot be empty.")
                    self.track_ingredient_recall(ingredient_input, user_date)
                    input("\nPress Enter to return to the menu...")
                    
                elif choice == '2':
                    station_input = None
                    while not station_input:
                        station_input = input("\nEnter the Station ID to map (e.g., cook_01): ").strip()
                        if not station_input:
                            print("Error: The input cannot be empty.")
                    self.map_contamination_risk(station_input, user_date)
                    input("\nPress Enter to return to the menu...")
                    
            else:
                print("\nInvalid choice. Please enter 1, 2, or 3.")

    def close(self):
        self.driver.close()

if __name__ == "__main__":
    app = Neo4jQueries()
    app.run_menu()
    app.close()


       NEO4J QUERY MENU
1. Track Ingredient Recall
2. Map Station Contamination Risk
3. Exit



Enter your choice (1/2/3):  2
Enter the date to search (Format: YYYY-MM-DD):  2026-03-15

Enter the Station ID to map (e.g., cook_01):  cook_02



--- MAPPING RISK FOR STATION cook_02 ON 2026-03-15 ---

Found 14 exposed batches and their respective paths.
- BATCH-1002 -> Later Stations: [pack_02] delivered by Driver drv_01 to Canteen cant_03
- BATCH-1008 -> Later Stations: [pack_01] delivered by Driver drv_06 to Canteen cant_01
- BATCH-1010 -> Later Stations: [pack_03] delivered by Driver drv_02 to Canteen cant_02
- BATCH-1015 -> Later Stations: [None] delivered by Driver drv_03 to Canteen cant_02
- BATCH-1018 -> Later Stations: [pack_01] delivered by Driver drv_02 to Canteen cant_02
- BATCH-1026 -> Later Stations: [pack_02] delivered by Driver drv_02 to Canteen cant_02
- BATCH-1030 -> Later Stations: [pack_02] delivered by Driver drv_05 to Canteen cant_02
- BATCH-1034 -> Later Stations: [pack_02] delivered by Driver drv_04 to Canteen cant_04
- BATCH-1037 -> Later Stations: [pack_01] delivered by Driver drv_07 to Canteen cant_04
- BATCH-1043 -> Later Stations: [None] delivered by Driver drv_04 to Canteen cant_04
- BATCH-1048 -> 


Press Enter to return to the menu... 



       NEO4J QUERY MENU
1. Track Ingredient Recall
2. Map Station Contamination Risk
3. Exit


...


--- MAPPING RISK FOR STATION pack_01 ON 2026-03-15 ---

Found 25 exposed batches and their respective paths.
- BATCH-1001 -> Later Stations: [None] delivered by Driver drv_02 to Canteen cant_01
- BATCH-1004 -> Later Stations: [None] delivered by Driver drv_05 to Canteen cant_02
- BATCH-1008 -> Later Stations: [None] delivered by Driver drv_06 to Canteen cant_01
- BATCH-1009 -> Later Stations: [None] delivered by Driver drv_01 to Canteen cant_02
- BATCH-1011 -> Later Stations: [None] delivered by Driver drv_05 to Canteen cant_04
- BATCH-1016 -> Later Stations: [None] delivered by Driver drv_06 to Canteen cant_03
- BATCH-1017 -> Later Stations: [None] delivered by Driver drv_01 to Canteen cant_04
- BATCH-1018 -> Later Stations: [None] delivered by Driver drv_02 to Canteen cant_02
- BATCH-1020 -> Later Stations: [None] delivered by Driver drv_03 to Canteen cant_03
- BATCH-1023 -> Later Stations: [None] delivered by Driver drv_06 to Canteen cant_01
- BATCH-1025 -> Later Stations: [None] d


Press Enter to return to the menu... 



       NEO4J QUERY MENU
1. Track Ingredient Recall
2. Map Station Contamination Risk
3. Exit



Enter your choice (1/2/3):  3


Exiting the query tool.
